In this notebook, the data from the 6 parquet files will be loaded and combined into one dataframe. The dataframe will only contains instances where the model is vicuna-13B. 

Due to the licensing of the data, the raw datafiles are not included in the public repository. However, there are detailed instructions in the readme on how to obtain the raw data.

In [6]:
import pandas as pd

# list of file names (parquet files)
file_names = ["train-00000-of-00006-4feeb3f83346a0e9.parquet",
            "train-00001-of-00006-4030672591c2f478.parquet",
            "train-00002-of-00006-1779b7cec9462180.parquet",
            "train-00003-of-00006-2fa862bfed56af1f.parquet",
            "train-00004-of-00006-18f4bdd50c103e71.parquet",
            "train-00005-of-00006-fe1acc5d10a9f0e2.parquet"]

dataframes = {}
# create a dataframe for each file
for i, name in enumerate(file_names):
    df = pd.read_parquet(f"../data/raw/{name}")
    dataframes[f"Dataset {i + 1}"] = df

print(dataframes)
    

{'Dataset 1':                          conversation_id         model  \
0       33f01939a744455c869cb234afca47f1  wizardlm-13b   
1       1e230e55efea4edab95db9cb87f6a9cb    vicuna-13b   
2       0f623736051f4a48a506fd5933563cfd    vicuna-13b   
3       e5c923a7fa3f4893beb432b4a06ef222        palm-2   
4       8ad66650dced4b728de1d14bb04657c1    vicuna-13b   
...                                  ...           ...   
166662  fc4468ca133545098f0b525e8f2be15c    alpaca-13b   
166663  7efdba6114474cb7b69d1a77e178e0d2     koala-13b   
166664  28650fe6328e40b2ad44af7f93b88d76     koala-13b   
166665  38666acffce04250b15bdbbcc432c019    vicuna-13b   
166666  92a9ed35a4704cb88e1082accdcd6b3f     koala-13b   

                                             conversation  turn language  \
0       [{'content': 'how can identity protection serv...     1  English   
1       [{'content': 'Beside OFAC's selective sanction...     6  English   
2       [{'content': 'You are the text completion mode...    

Examine one singular dataframe

In [7]:
df1 = dataframes["Dataset 1"]

df1.head()

,conversation_id,model,conversation,turn,language,openai_moderation,redacted
0,33f01939a744455c869cb234afca47f1,wizardlm-13b,[{'content': 'how can identity protection serv...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
1,1e230e55efea4edab95db9cb87f6a9cb,vicuna-13b,[{'content': 'Beside OFAC's selective sanction...,6,English,"[{'categories': {'harassment': False, 'harassm...",False
2,0f623736051f4a48a506fd5933563cfd,vicuna-13b,[{'content': 'You are the text completion mode...,1,English,"[{'categories': {'harassment': False, 'harassm...",False
3,e5c923a7fa3f4893beb432b4a06ef222,palm-2,[{'content': 'The sum of the perimeters of thr...,2,English,"[{'categories': {'harassment': False, 'harassm...",False
4,8ad66650dced4b728de1d14bb04657c1,vicuna-13b,[{'content': 'What is the type of the variable...,1,English,"[{'categories': {'harassment': False, 'harassm...",False


In [12]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166667 entries, 0 to 166666
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   conversation_id    166667 non-null  object
 1   model              166667 non-null  object
 2   conversation       166667 non-null  object
 3   turn               166667 non-null  int64 
 4   language           166667 non-null  object
 5   openai_moderation  166667 non-null  object
 6   redacted           166667 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


In [13]:
df1["model"].value_counts()

model
vicuna-13b                 81635
koala-13b                  13600
alpaca-13b                 10309
chatglm-6b                  6003
llama-13b                   5341
vicuna-33b                  5240
llama-2-13b-chat            5070
fastchat-t5-3b              4466
oasst-pythia-12b            4362
dolly-v2-12b                3824
claude-1                    3796
wizardlm-13b                2817
vicuna-7b                   2750
RWKV-4-Raven-14B            2625
mpt-7b-chat                 2515
guanaco-33b                 2314
stablelm-tuned-alpha-7b     1976
mpt-30b-chat                1469
gpt-3.5-turbo               1291
gpt4all-13b-snoozy          1245
gpt-4                       1222
palm-2                       931
claude-instant-1             916
llama-2-7b-chat              590
claude-2                     360
Name: count, dtype: int64

Vicuna-13b is the chosen model for two reasons:

1. It is the model with the most training examples.
2. It's tokenizer is public. Since the dataset does not contain output tokens, which is our eventual target, we need to use the publicly available tokenizer to calculate the output tokens. Most of these models do not have public tokenizers.

In [14]:
filtered_datasets = {}

for i, dataset in enumerate(dataframes.values()):
    filtered = dataset[dataset["model"] == "vicuna-13b"]
    print(f"Number of instances in dataset {i + 1}: {len(filtered)}")
    filtered_datasets[f"Filtered Dataset {i + 1}"] = filtered

Number of instances in dataset 1: 81635
Number of instances in dataset 2: 81673
Number of instances in dataset 3: 82072
Number of instances in dataset 4: 81912
Number of instances in dataset 5: 81407
Number of instances in dataset 6: 82013


In [15]:
combined_df = pd.concat([df for df in filtered_datasets.values()], ignore_index=True)
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490712 entries, 0 to 490711
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   conversation_id    490712 non-null  object
 1   model              490712 non-null  object
 2   conversation       490712 non-null  object
 3   turn               490712 non-null  int64 
 4   language           490712 non-null  object
 5   openai_moderation  490712 non-null  object
 6   redacted           490712 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 22.9+ MB


Few checks to ensure merge was completely succesfully
1. Check number of instances in each dataset equals total instances in combined dataset
2. Check model type only has one value: vicuna-13b

In [17]:
print(490712 == 81635 + 81673 + 82072 + 81912 + 81407 + 82013)

True


In [18]:
combined_df["model"].value_counts()

model
vicuna-13b    490712
Name: count, dtype: int64

Merge looks successful. Now exporting the data as parquet for use in other notebooks.

In [21]:
combined_df.to_parquet("../data/processed/combined_filtered_dataset.parquet", index=False)